# Baliser le corpus avec Stanza

## Objectif

Ce notebook ajoute des balises TEI aux entités nommées détectées dans les fichiers XML du corpus.

Les modèles Stanza français et italien sont utilisés pour reconnaître les personnes et les lieux. Les fichiers originaux sont conservés : les résultats sont écrits dans un dossier séparé.

## 1. Importer les bibliothèques

- `stanza` fournit les pipelines de reconnaissance d'entités nommées ;
- `lxml` permet de lire et modifier l'arbre XML ;
- `pathlib` gère les chemins de fichiers ;
- `joblib` distribue le traitement des fichiers entre plusieurs processus ;
- `re` sert à choisir heuristiquement le modèle français ou italien.

## 2. Télécharger les modèles Stanza

Cette cellule prépare les ressources nécessaires pour les langues française et italienne. Elle n'est à exécuter qu'une première fois, ou lorsqu'une ressource locale est absente.

Les modèles sont stockés dans `C:/stanza_models`, puis réutilisés avec `REUSE_RESOURCES` dans le traitement. Le téléchargement n'est pas effectué par les workers.

In [ ]:
import re
from pathlib import Path

import stanza
from joblib import Parallel, delayed
from lxml import etree
from stanza.pipeline.core import DownloadMethod

In [23]:
# Téléchargement des modèles Stanza pour le français et l'italien
# À n'exécuter qu'une seule fois
stanza.download('fr', model_dir='C:/stanza_models')
stanza.download('it', model_dir='C:/stanza_models')

2026-03-19 12:06:10 INFO: Downloaded file to C:/stanza_models\resources.json
2026-03-19 12:06:10 INFO: Downloading default packages for language: fr (French) ...
2026-03-19 12:06:10 INFO: File exists: C:/stanza_models\fr\default.zip
2026-03-19 12:06:12 INFO: Finished downloading models and saved to C:/stanza_models


2026-03-19 12:06:12 INFO: Downloaded file to C:/stanza_models\resources.json
2026-03-19 12:06:12 INFO: Downloading default packages for language: it (Italian) ...
2026-03-19 12:06:13 INFO: File exists: C:/stanza_models\it\default.zip
2026-03-19 12:06:16 INFO: Finished downloading models and saved to C:/stanza_models


[['zip', 'default.zip']]

## 3. Définir les chemins et les balises TEI

Les fichiers sources sont lus dans `output/v0/`. Les fichiers annotés sont écrits dans `output/Vstanza/` afin de préserver les originaux.

`LABEL_MAP` convertit les catégories produites par Stanza en balises attendues par le corpus : les personnes deviennent `persName`, tandis que les lieux et entités géographiques deviennent `placeName`.

In [24]:
DIR_V0 = Path('output/v0/')
OUTPUT_DIR = Path('output/Vstanza/')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Correspondance entre les labels Stanza et les balises TEI
LABEL_MAP = {
    "PER":  "persName",
    "LOC":  "placeName",
    "GPE":  "placeName",
}

## 4. Détecter les entités et les injecter dans le XML

Pour chaque portion de texte, le script choisit le modèle français ou italien, puis conserve uniquement les entités `PER`, `LOC` et `GPE`.

Stanza fournit les positions `start_char` et `end_char`. Ces offsets permettent de reprendre exactement les sous-chaînes originales, sans perdre les espaces ni modifier le texte non reconnu.

Dans `lxml`, le texte avant le premier enfant est stocké dans `element.text`, tandis que le texte après un enfant est stocké dans `child.tail`. Les deux emplacements doivent être traités pour annoter tout le contenu du corpus.

Les enfants présents avant les insertions sont parcourus avec une position contrôlée : les balises ajoutées ne doivent pas être réanalysées comme du texte source.

In [ ]:
def detect_lang(text):
    """Choisit le modèle à partir de marqueurs français ou italiens."""
    fr_markers = len(re.findall(
        r"\b(le|la|les|de|du|des|un|une|et|en|je|il|elle|nous|vous|ils)\b",
        text,
        re.IGNORECASE,
    ))
    it_markers = len(re.findall(
        r"\b(il|la|le|di|del|della|un|una|e|in|io|lui|lei|noi|voi|loro)\b",
        text,
        re.IGNORECASE,
    ))
    return "it" if it_markers > fr_markers else "fr"


def get_entities(text, nlp_fra, nlp_ita):
    """Retourne les entités utiles avec leurs offsets et leurs balises TEI."""
    # Un segment vide ne nécessite pas d'appel au pipeline Stanza.
    if not text or not text.strip():
        return []

    # Le modèle est choisi segment par segment, car le corpus peut être multilingue.
    nlp = nlp_ita if detect_lang(text) == "it" else nlp_fra
    doc = nlp(text)

    entities = []
    for ent in doc.ents:
        tag = LABEL_MAP.get(ent.type)
        # Les catégories absentes de LABEL_MAP restent inchangées dans le XML.
        if tag:
            entities.append((ent.start_char, ent.end_char, tag))
    return entities


def inject_entities(parent, original_text, entities, insert_before_index):
    """Remplace un texte par du texte brut et des éléments TEI balisés."""
    if not entities:
        return 0

    # Les offsets Stanza permettent de conserver exactement les sous-chaînes
    # originales, y compris les espaces et la ponctuation.
    segments = []
    last = 0
    for start, end, tag in entities:
        if start > last:
            segments.append((original_text[last:start], None))
        segments.append((original_text[start:end], tag))
        last = end
    if last < len(original_text):
        segments.append((original_text[last:], None))

    # Dans lxml, le texte avant un enfant est dans .text ou dans le .tail
    # de l'enfant précédent. Le texte après une balise insérée va dans son .tail.
    inserted = 0
    pending_text = ""
    if insert_before_index == 0:
        parent.text = ""
    else:
        parent[insert_before_index - 1].tail = ""

    for text_chunk, tag in segments:
        if tag is None:
            pending_text += text_chunk
            continue

        # Placer le texte brut avant l'entité, puis insérer l'entité à sa place.
        pos = insert_before_index + inserted
        if pos == 0:
            parent.text = (parent.text or "") + pending_text
        else:
            parent[pos - 1].tail = (parent[pos - 1].tail or "") + pending_text
        pending_text = ""

        new_el = etree.Element(tag)
        new_el.text = text_chunk
        new_el.tail = ""
        parent.insert(pos, new_el)
        inserted += 1

    # Conserver le texte situé après la dernière entité.
    if pending_text:
        pos = insert_before_index + inserted
        if pos == 0:
            parent.text = (parent.text or "") + pending_text
        else:
            parent[pos - 1].tail = (parent[pos - 1].tail or "") + pending_text

    return inserted


def process_element(root, nlp_fra, nlp_ita):
    """Parcourt l'arbre et traite les zones textuelles non encore balisées."""
    # Le parcours est préparé avant les insertions pour distinguer le XML source
    # des éléments créés par le script.
    all_elements = []
    stack = [root]
    while stack:
        element = stack.pop()
        all_elements.append(element)
        for child in reversed(list(element)):
            stack.append(child)

    for element in all_elements:
        try:
            # Texte situé avant le premier enfant de l'élément.
            if element.text and element.text.strip():
                entities = get_entities(element.text, nlp_fra, nlp_ita)
                if entities:
                    inject_entities(element, element.text, entities, insert_before_index=0)

            # Texte situé après chaque enfant. L'indice avance pour ignorer les
            # nouveaux éléments déjà produits par inject_entities().
            index = 0
            while index < len(element):
                child = element[index]
                index += 1
                if child.tail and child.tail.strip():
                    entities = get_entities(child.tail, nlp_fra, nlp_ita)
                    if entities:
                        tail_text = child.tail
                        child.tail = None
                        inserted = inject_entities(
                            element,
                            tail_text,
                            entities,
                            insert_before_index=index,
                        )
                        index += inserted

        except Exception as error:
            # Une erreur locale est signalée sans interrompre les autres éléments.
            import traceback
            print(f"  → Erreur sur élément <{element.tag}> : {error}")
            traceback.print_exc()
            continue


def process_file(xml_path, output_dir):
    """Traite un fichier XML dans un worker et écrit sa copie annotée."""
    # Les pipelines sont créés dans le worker : ils sont volumineux et ne sont
    # pas sérialisés pour être transmis directement à joblib.
    nlp_fra = stanza.Pipeline(
        "fr",
        model_dir="C:/stanza_models",
        processors="tokenize,ner",
        use_gpu=False,
        verbose=False,
        download_method=DownloadMethod.REUSE_RESOURCES,
    )
    nlp_ita = stanza.Pipeline(
        "it",
        model_dir="C:/stanza_models",
        processors="tokenize,ner",
        use_gpu=False,
        verbose=False,
        download_method=DownloadMethod.REUSE_RESOURCES,
    )

    try:
        # remove_blank_text=False évite de supprimer les espaces existants.
        tree = etree.parse(
            str(xml_path),
            etree.XMLParser(remove_blank_text=False),
        )
        process_element(tree.getroot(), nlp_fra, nlp_ita)
        output_path = output_dir / xml_path.name
        tree.write(
            str(output_path),
            encoding="utf-8",
            xml_declaration=True,
            pretty_print=True,
        )
        return f"✓ {xml_path.name}"
    except Exception as error:
        return f"✗ {xml_path.name} — erreur : {error}"

## 5. Traiter les fichiers en parallèle

Chaque fichier est confié à un worker indépendant. Le pipeline Stanza étant volumineux et non sérialisable, il est créé dans `process_file()` pour chaque worker.

Le nombre de workers est fixé à quatre, mais peut être réduit si la mémoire disponible est limitée.

In [30]:
# --- Exécution parallèle ---
# Stanza est gourmand en mémoire : réduire n_jobs si nécessaire
xml_files = list(DIR_V0.glob('*.xml'))
print(f"{len(xml_files)} fichiers à traiter avec 4 workers...\n")

results = Parallel(n_jobs=4, backend="loky", verbose=0)(
    delayed(process_file)(f, OUTPUT_DIR) for f in xml_files
)

for r in results:
    print(r)
print("\nTerminé.")

19 fichiers à traiter avec 4 workers...



c:\Users\ebondoer\anaconda3\envs\stanza-env\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


✓ Agucchi_TrattatoPittura.xml
✓ Daret_VieRaphael.xml
✓ DupuyDuGrez_TraitePeinture.xml
✓ Freart_IdeaDellaPerfezione.xml
✓ Lomazzo_Idea.xml
✓ Lomazzo_TraicteProportion.xml
✓ Marino_DicerieSacre.xml
✓ Monier_HistoireArtsRapportDessein.xml
✓ Pader_LaPeintureParlante.xml
✓ Pader_SongeEnigmatique.xml
✓ Piles_AbregeViePeintres.xml
✓ Piles_ConversationsConnaissancePeinture.xml
✓ Piles_CoursPeinture.xml
✓ Piles_DialogueColoris.xml
✓ Vinci_TraitePeinture_fra.xml
✓ Vinci_TrattatoPittura_ITA.xml
✓ Zuccari_IdeaPittori.xml
✓ Zuccari_Lettera.xml
✓ Zuccari_OrigineProgressoAcademiaDissegno.xml

Terminé.
